# ML-Based Optimization for RVG Unified Field Propulsion

This interactive Jupyter notebook demonstrates machine learning applications for the **Refractive Vacuum Gravity (RVG) Unified Field** framework:

- **∇B² Gradient Optimization**: Using TensorFlow to optimize magnetic field gradients for maximum thrust via the Master Equation of Levitation
- **Dilaton Enhancement Θ_dilaton(B) Fitting**: ML-based calibration of the critical non-linear vacuum response function
- **MADA Convergence Optimization**: Optimize field vector directions to maximize convergence quality (prevents configuration errors)
- **Supra-Saturation Regime Optimization**: Find optimal B/B_sat operating points for different materials
- **Predictive Maintenance**: Neural network for material degradation prediction including MADA convergence quality

**Framework References:**
- [RVG Unified Field Theory](https://dx.doi.org/10.2139/ssrn.5381654) (Hofseth, 2025)
- [U.S. Patent #5,929,732 - MADA](https://patents.google.com/patent/US5929732A/en) (Lockheed Martin)
- CMS/ATLAS 95.4 GeV di-photon resonance (3.1σ combined significance)

**Key Equations:**
- Master Equation: $\mathbf{F}_{\text{lift}} = \int_V \frac{1}{2\mu_0} \Theta_{\text{dilaton}}(B) \cdot \nabla B^2 \, dV$
- Vacuum Refractive Index: $K(\mathbf{r}) = 1 + \Theta_{95} \frac{B^2}{B_{\text{crit}}^2}$
- Dilaton Enhancement: $\Theta_{\text{dilaton}}(B) = \theta_{\text{base}} \cdot (1 + (B/B_{\text{crit}})^2) \cdot f_{\text{activation}}(B)$

Run cells sequentially. Adjust hyperparameters for experimentation.

In [ ]:
import sys
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Input, Concatenate
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Add parent directory to path for imports
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), '..'))

# Try importing from project modules
try:
    from simulations.equations import force_vector
    from ai.navigation import MaintenanceNN
    MODULES_AVAILABLE = True
except ImportError:
    MODULES_AVAILABLE = False
    print("Project modules not found. Using standalone implementations.")

print("Libraries imported successfully")
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

## RVG Framework Constants and Functions

Define the key physical constants and functions from the RVG Unified Field framework.

In [ ]:
# =============================================================================
# RVG UNIFIED FIELD CONSTANTS
# =============================================================================

# Fundamental constants
MU_0 = 4 * np.pi * 1e-7  # Vacuum permeability (H/m)
EPSILON_0 = 8.854187817e-12  # Vacuum permittivity (F/m)
C = 299792458.0  # Speed of light (m/s)

# 95 GeV Dilaton/Radion Resonance Parameters
DILATON_MASS = 95.4  # GeV - observed CMS/ATLAS resonance
DILATON_SIGMA = 3.1  # Combined significance (σ)

# Default RVG parameters (require experimental calibration)
DEFAULT_THETA_BASE = 1e-6  # Base dilaton enhancement (placeholder)
DEFAULT_B_CRIT = 20.0  # Effective critical field for activation (T)

# Material saturation limits (from Materials Ranking)
B_SAT_MINNEALLOY = 2.85  # Minnealloy α'-Fe₈(NC) saturation (T) - BEST
B_SAT_HIPERCO = 2.4  # Hiperco-50 saturation (T)
B_SAT_IRON = 2.1  # Pure iron (ARMCO) saturation (T)

# MADA Amplification (per U.S. Patent 5,929,732)
MADA_K_DEFAULT = 200.0  # Default amplification (~200x vs single magnet)
MADA_K_MAX = 529.0  # Maximum theoretical amplification

# Simulation parameters
DEFAULT_VOLUME = 0.1  # Integration volume (m³)
DEFAULT_ETA = 0.95  # Alignment efficiency

# Convergence quality thresholds
CONVERGENCE_OPTIMAL = 0.95
CONVERGENCE_WARNING = 0.85
CONVERGENCE_CRITICAL = 0.70

print("RVG Framework Constants Loaded:")
print(f"  Dilaton mass: {DILATON_MASS} GeV ({DILATON_SIGMA}σ significance)")
print(f"  Default θ_base: {DEFAULT_THETA_BASE}")
print(f"  Default B_crit: {DEFAULT_B_CRIT} T")
print(f"  MADA amplification: {MADA_K_DEFAULT}-{MADA_K_MAX}x")
print(f"  Material B_sat: Minnealloy={B_SAT_MINNEALLOY}T, Hiperco={B_SAT_HIPERCO}T, Iron={B_SAT_IRON}T")

In [ ]:
# =============================================================================
# RVG CORE FUNCTIONS (TensorFlow-compatible)
# =============================================================================

def theta_dilaton_simple_tf(B, theta_base, B_crit):
    """
    Simple dilaton enhancement model (TensorFlow).
    Θ_dilaton(B) = θ_base * (1 + (B / B_crit)²)
    """
    ratio = B / (B_crit + 1e-10)
    return theta_base * (1.0 + ratio**2)


def theta_dilaton_resonance_tf(B, theta_base, B_crit, gamma, epsilon):
    """
    Dilaton enhancement with 95 GeV resonance activation (TensorFlow).
    Θ_dilaton(B) = θ_base * (1 + (B/B_crit)²) * exp(-γ / (B/B_crit + ε))
    """
    ratio = B / (B_crit + 1e-10)
    activation = tf.exp(-gamma / (ratio + epsilon))
    return theta_base * (1.0 + ratio**2) * activation


def vacuum_refractive_index_tf(B, theta_base, B_crit):
    """
    Vacuum refractive index K(r) (TensorFlow).
    K = 1 + Θ_dilaton(B) * (B / B_crit)²
    """
    theta = theta_dilaton_simple_tf(B, theta_base, B_crit)
    chi_vac = theta * (B / (B_crit + 1e-10))**2
    return 1.0 + chi_vac


def master_equation_thrust_tf(B, grad_B2, volume, theta_base, B_crit, eta):
    """
    Master Equation of Levitation thrust (TensorFlow).
    F_lift = (1 / 2μ₀) * Θ_dilaton(B) * ∇(B²) * V * η
    """
    theta = theta_dilaton_simple_tf(B, theta_base, B_crit)
    F = (1.0 / (2.0 * MU_0)) * theta * grad_B2 * volume * eta
    return F


def supra_saturation_factor_tf(B_opposing, B_sat, n=2.0):
    """
    Supra-saturation effectiveness factor (TensorFlow).
    Effects manifest when B_opposing >> B_sat.
    """
    ratio = B_opposing / (B_sat + 1e-10)
    # Effectiveness grows with supra-saturation ratio
    effectiveness = tf.where(
        ratio > 1.0,
        tf.minimum(ratio**n / 25.0, 1.0),  # Saturates at ratio^n/25
        ratio**(n/2) / 25.0  # Reduced below saturation
    )
    return effectiveness


def calculate_convergence_quality(B1_vec, B2_vec):
    """
    Calculate MADA convergence quality from field vectors.
    Returns 1.0 for perfect opposition, -1.0 for parallel (wrong!).
    """
    B1_norm = B1_vec / (np.linalg.norm(B1_vec) + 1e-10)
    B2_norm = B2_vec / (np.linalg.norm(B2_vec) + 1e-10)
    # Quality = negative dot product (opposing = positive quality)
    return -np.dot(B1_norm, B2_norm)


print("RVG Core Functions Defined:")
print("  - theta_dilaton_simple_tf(B, θ_base, B_crit)")
print("  - theta_dilaton_resonance_tf(B, θ_base, B_crit, γ, ε)")
print("  - vacuum_refractive_index_tf(B, θ_base, B_crit)")
print("  - master_equation_thrust_tf(B, ∇B², V, θ_base, B_crit, η)")
print("  - supra_saturation_factor_tf(B_opposing, B_sat, n)")
print("  - calculate_convergence_quality(B1_vec, B2_vec)")

## Part 1: Master Equation ∇B² Gradient Optimization

Optimize the magnetic field gradient ∇B² to maximize thrust via the Master Equation of Levitation:

$$\mathbf{F}_{\text{lift}} = \frac{1}{2\mu_0} \Theta_{\text{dilaton}}(B) \cdot \nabla B^2 \cdot V \cdot \eta$$

We treat ∇B² components as trainable variables and maximize thrust magnitude.

In [ ]:
# =============================================================================
# PART 1: ∇B² GRADIENT OPTIMIZATION
# =============================================================================

# Fixed parameters
B_FIELD = 50.0  # Operating B-field (T)
VOLUME = 0.1  # Integration volume (m³)
THETA_BASE = 1e-6  # Dilaton base (to be calibrated experimentally)
B_CRIT = 20.0  # Critical field (T)
ETA = 0.95  # Alignment efficiency

# Trainable gradient ∇B² (3D vector in T²/m)
# Initial: pointing in +x direction with moderate magnitude
initial_grad_B2 = tf.Variable([1e9, 0.0, 0.0], dtype=tf.float32, name='grad_B2')

# Optimizer
optimizer = Adam(learning_rate=1e8)  # Large LR for gradient scaling

# Loss function: Negative thrust magnitude (to maximize)
def thrust_loss_fn(grad_B2):
    """
    Loss = -|F_lift| to maximize thrust via gradient descent.
    """
    grad_B2_magnitude = tf.norm(grad_B2)
    thrust = master_equation_thrust_tf(
        B_FIELD, grad_B2_magnitude, VOLUME, THETA_BASE, B_CRIT, ETA
    )
    return -thrust  # Negative for maximization

# Training loop
thrust_history = []
grad_history = []
theta_history = []

print("Optimizing ∇B² for maximum thrust via Master Equation...")
print(f"Fixed: B={B_FIELD}T, V={VOLUME}m³, θ_base={THETA_BASE}, B_crit={B_CRIT}T, η={ETA}")
print("="*60)

for step in range(100):
    with tf.GradientTape() as tape:
        loss = thrust_loss_fn(initial_grad_B2)
    
    grads = tape.gradient(loss, [initial_grad_B2])
    optimizer.apply_gradients(zip(grads, [initial_grad_B2]))
    
    # Record history
    current_thrust = -loss.numpy()
    current_grad = initial_grad_B2.numpy().copy()
    current_theta = theta_dilaton_simple_tf(B_FIELD, THETA_BASE, B_CRIT).numpy()
    
    thrust_history.append(current_thrust)
    grad_history.append(current_grad)
    theta_history.append(current_theta)
    
    if step % 20 == 0:
        grad_mag = np.linalg.norm(current_grad)
        print(f"Step {step:3d}: Thrust = {current_thrust:.4e} N, |∇B²| = {grad_mag:.2e} T²/m")

print("="*60)
print(f"\nOptimization Complete!")
print(f"Optimal ∇B²: [{initial_grad_B2.numpy()[0]:.2e}, {initial_grad_B2.numpy()[1]:.2e}, {initial_grad_B2.numpy()[2]:.2e}] T²/m")
print(f"Final thrust: {thrust_history[-1]:.4e} N")
print(f"Θ_dilaton at B={B_FIELD}T: {theta_history[-1]:.2e}")

In [ ]:
# Visualization: Thrust optimization progress
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: Thrust vs Step
axes[0].plot(thrust_history, linewidth=2, color='blue')
axes[0].set_xlabel('Optimization Step', fontsize=12)
axes[0].set_ylabel('Thrust (N)', fontsize=12)
axes[0].set_title('Master Equation Thrust Optimization', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# Middle: ∇B² magnitude vs Step
grad_mags = [np.linalg.norm(g) for g in grad_history]
axes[1].plot(grad_mags, linewidth=2, color='green')
axes[1].set_xlabel('Optimization Step', fontsize=12)
axes[1].set_ylabel('|∇B²| (T²/m)', fontsize=12)
axes[1].set_title('Gradient Magnitude Evolution', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')

# Right: Thrust vs ∇B² relationship
axes[2].scatter(grad_mags, thrust_history, c=range(len(thrust_history)), cmap='viridis', alpha=0.7)
axes[2].set_xlabel('|∇B²| (T²/m)', fontsize=12)
axes[2].set_ylabel('Thrust (N)', fontsize=12)
axes[2].set_title('Thrust vs Gradient (colored by step)', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].set_xscale('log')
axes[2].set_yscale('log')

plt.tight_layout()
plt.show()

print("\n📊 Insight: Thrust scales linearly with |∇B²| per Master Equation.")
print("   Higher gradients (>10¹⁰ T²/m) require MADA supra-saturation engineering.")

## Part 2: Dilaton Enhancement Θ_dilaton(B) Fitting

**This is the most critical calibration for the RVG framework!**

Train a neural network to fit the dilaton enhancement function Θ_dilaton(B) from synthetic or experimental data.
The goal is to learn the relationship:

$$\Theta_{\text{dilaton}}(B) = \theta_{\text{base}} \cdot \left(1 + \alpha_T \frac{B^2}{B_{\text{crit}}^2} + \beta_T \frac{B^4}{B_{\text{crit}}^4}\right) \cdot f_{\text{res}}(B)$$

In [ ]:
# =============================================================================
# PART 2: DILATON ENHANCEMENT Θ_dilaton(B) FITTING
# =============================================================================

# Generate synthetic Θ_dilaton data (simulating experimental measurements)
# In practice, this would come from thrust measurements and inference

np.random.seed(42)

# True parameters (what we're trying to recover)
TRUE_THETA_BASE = 1.2e-6
TRUE_B_CRIT = 18.5
TRUE_GAMMA = 0.08
TRUE_EPSILON = 0.02

# Generate B-field range (5 to 80 T)
n_samples = 200
B_data = np.random.uniform(5, 80, n_samples)

# True Θ values using resonance model + noise
def theta_true(B):
    ratio = B / TRUE_B_CRIT
    activation = np.exp(-TRUE_GAMMA / (ratio + TRUE_EPSILON))
    return TRUE_THETA_BASE * (1 + ratio**2) * activation

theta_data_true = theta_true(B_data)
noise_level = 0.1  # 10% noise
theta_data = theta_data_true * (1 + np.random.normal(0, noise_level, n_samples))

print("Generated synthetic Θ_dilaton calibration data:")
print(f"  Samples: {n_samples}")
print(f"  B range: {B_data.min():.1f} - {B_data.max():.1f} T")
print(f"  Θ range: {theta_data.min():.2e} - {theta_data.max():.2e}")
print(f"  Noise level: {noise_level*100:.0f}%")
print(f"\nTrue parameters (to recover):")
print(f"  θ_base = {TRUE_THETA_BASE:.2e}")
print(f"  B_crit = {TRUE_B_CRIT} T")
print(f"  γ = {TRUE_GAMMA}")
print(f"  ε = {TRUE_EPSILON}")

In [ ]:
# Build neural network to fit Θ_dilaton(B)
# Network learns: B -> Θ_dilaton

# Normalize inputs
B_mean, B_std = B_data.mean(), B_data.std()
theta_mean, theta_std = theta_data.mean(), theta_data.std()

B_normalized = (B_data - B_mean) / B_std
theta_normalized = (theta_data - theta_mean) / theta_std

# Split data
split_idx = int(0.8 * n_samples)
B_train, B_test = B_normalized[:split_idx], B_normalized[split_idx:]
theta_train, theta_test = theta_normalized[:split_idx], theta_normalized[split_idx:]

# Build model
theta_model = Sequential([
    Dense(64, activation='relu', input_shape=(1,)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])

theta_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

print("Neural Network for Θ_dilaton(B):")
theta_model.summary()

# Train
print("\nTraining...")
history = theta_model.fit(
    B_train, theta_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=0
)

# Evaluate
test_loss, test_mae = theta_model.evaluate(B_test, theta_test, verbose=0)
print(f"\nTest Loss (MSE): {test_loss:.6f}")
print(f"Test MAE: {test_mae:.6f}")

In [ ]:
# Visualization: Θ_dilaton fitting results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: Training history
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('MSE Loss', fontsize=12)
axes[0].set_title('Θ_dilaton Network Training', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Middle: Predicted vs True
B_fine = np.linspace(5, 80, 200)
B_fine_norm = (B_fine - B_mean) / B_std
theta_pred_norm = theta_model.predict(B_fine_norm, verbose=0)
theta_pred = theta_pred_norm * theta_std + theta_mean
theta_true_fine = theta_true(B_fine)

axes[1].scatter(B_data, theta_data, alpha=0.3, label='Noisy Data', s=20)
axes[1].plot(B_fine, theta_true_fine, 'r-', linewidth=2, label='True Model')
axes[1].plot(B_fine, theta_pred, 'b--', linewidth=2, label='NN Prediction')
axes[1].set_xlabel('Magnetic Field B (T)', fontsize=12)
axes[1].set_ylabel('Θ_dilaton', fontsize=12)
axes[1].set_title('Dilaton Enhancement Fitting', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')

# Right: Residuals
theta_pred_test = theta_model.predict(B_test, verbose=0) * theta_std + theta_mean
theta_test_denorm = theta_test * theta_std + theta_mean
residuals = theta_pred_test.flatten() - theta_test_denorm

axes[2].hist(residuals / theta_test_denorm * 100, bins=20, edgecolor='black', alpha=0.7)
axes[2].axvline(x=0, color='r', linestyle='--')
axes[2].set_xlabel('Relative Error (%)', fontsize=12)
axes[2].set_ylabel('Frequency', fontsize=12)
axes[2].set_title('Prediction Residuals', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 The NN successfully learns the non-linear Θ_dilaton(B) relationship.")
print("   In practice, train on real thrust data using inferred Θ values.")

## Part 3: MADA Convergence Optimization

**CRITICAL: This prevents configuration errors like reversed field directions!**

Optimize MADA field vector directions to maximize convergence quality.
Starting from potentially incorrect directions (fields pointing away), the optimizer finds the correct opposing configuration.

Loss function: -convergence_quality (to maximize quality)

In [ ]:
# =============================================================================
# PART 3: MADA CONVERGENCE OPTIMIZATION
# =============================================================================

# MADA positions (fixed Bushman opposing configuration)
mada1_pos = np.array([-0.3, 0.0, 0.0])  # Left side
mada2_pos = np.array([0.3, 0.0, 0.0])   # Right side
center = np.array([0.0, 0.0, 0.0])      # Focal/convergence point

# Field magnitude (T)
B_magnitude = 50.0

# Start with WRONG configuration (both pointing away from center!)
# This simulates a configuration error that should be detected and corrected
B1_direction_initial = (mada1_pos - center) / np.linalg.norm(mada1_pos - center)  # WRONG!
B2_direction_initial = (mada2_pos - center) / np.linalg.norm(mada2_pos - center)  # WRONG!

print("MADA CONVERGENCE OPTIMIZATION")
print("="*60)
print("\nInitial (INCORRECT) configuration:")
print(f"  B1 direction: {B1_direction_initial} (pointing AWAY from center!)")
print(f"  B2 direction: {B2_direction_initial} (pointing AWAY from center!)")

initial_quality = calculate_convergence_quality(
    B1_direction_initial * B_magnitude,
    B2_direction_initial * B_magnitude
)
print(f"  Initial convergence quality: {initial_quality:.3f}")

if initial_quality < CONVERGENCE_CRITICAL:
    print("  🔴 CRITICAL: Fields are DIVERGING! Configuration error detected!")
elif initial_quality < CONVERGENCE_WARNING:
    print("  ⚠️ WARNING: Poor convergence quality!")
else:
    print("  ✓ Convergence quality acceptable")

In [ ]:
# Trainable field direction variables
B1_dir = tf.Variable(B1_direction_initial.astype(np.float32), dtype=tf.float32)
B2_dir = tf.Variable(B2_direction_initial.astype(np.float32), dtype=tf.float32)

# Optimizer
convergence_optimizer = Adam(learning_rate=0.05)

# Loss function: Negative convergence quality
def convergence_loss_fn(B1_dir, B2_dir):
    # Normalize directions to unit vectors
    B1_norm = B1_dir / (tf.norm(B1_dir) + 1e-8)
    B2_norm = B2_dir / (tf.norm(B2_dir) + 1e-8)
    
    # Convergence quality = negative dot product
    # (opposing vectors have dot product = -1, so quality = +1)
    dot_product = tf.reduce_sum(B1_norm * B2_norm)
    quality = -dot_product
    
    # Loss = negative quality (to maximize)
    return -quality

# Training loop
convergence_losses = []
quality_history = []
B1_dir_history = []
B2_dir_history = []

print("\nOptimizing MADA field directions...")
print("-"*60)

for step in range(100):
    with tf.GradientTape() as tape:
        loss = convergence_loss_fn(B1_dir, B2_dir)
    
    grads = tape.gradient(loss, [B1_dir, B2_dir])
    convergence_optimizer.apply_gradients(zip(grads, [B1_dir, B2_dir]))
    
    # Calculate quality for logging
    B1_normalized = B1_dir.numpy() / (np.linalg.norm(B1_dir.numpy()) + 1e-8)
    B2_normalized = B2_dir.numpy() / (np.linalg.norm(B2_dir.numpy()) + 1e-8)
    quality = calculate_convergence_quality(
        B1_normalized * B_magnitude,
        B2_normalized * B_magnitude
    )
    
    convergence_losses.append(-loss.numpy())
    quality_history.append(quality)
    B1_dir_history.append(B1_normalized.copy())
    B2_dir_history.append(B2_normalized.copy())
    
    if step % 20 == 0:
        if quality >= CONVERGENCE_OPTIMAL:
            status = "✓ OPTIMAL"
        elif quality >= CONVERGENCE_WARNING:
            status = "✓ Good"
        elif quality >= CONVERGENCE_CRITICAL:
            status = "⚠️ Warning"
        else:
            status = "🔴 Critical"
        print(f"Step {step:3d}: Quality = {quality:+.3f} {status}")

# Final results
B1_final = B1_dir.numpy() / np.linalg.norm(B1_dir.numpy())
B2_final = B2_dir.numpy() / np.linalg.norm(B2_dir.numpy())
final_quality = quality_history[-1]

print("-"*60)
print(f"\nOptimization Complete!")
print(f"\nFinal (CORRECTED) configuration:")
print(f"  B1 direction: {B1_final}")
print(f"  B2 direction: {B2_final}")
print(f"  Final convergence quality: {final_quality:.3f}")

if final_quality >= CONVERGENCE_OPTIMAL:
    print(f"  ✓ OPTIMAL: Fields properly opposing for maximum thrust!")

# Expected directions (pointing toward center)
expected_B1 = (center - mada1_pos) / np.linalg.norm(center - mada1_pos)
expected_B2 = (center - mada2_pos) / np.linalg.norm(center - mada2_pos)
print(f"\nVerification:")
print(f"  Expected B1: {expected_B1}")
print(f"  Expected B2: {expected_B2}")
print(f"  B1 alignment error: {np.linalg.norm(B1_final - expected_B1):.4f}")
print(f"  B2 alignment error: {np.linalg.norm(B2_final - expected_B2):.4f}")

In [ ]:
# Visualization: Convergence optimization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: Convergence quality over time
steps = np.arange(len(quality_history))
ax1.plot(steps, quality_history, linewidth=2, color='blue', label='Convergence Quality')
ax1.axhline(y=CONVERGENCE_OPTIMAL, color='green', linestyle='--', label=f'Optimal ({CONVERGENCE_OPTIMAL})')
ax1.axhline(y=CONVERGENCE_WARNING, color='orange', linestyle='--', label=f'Warning ({CONVERGENCE_WARNING})')
ax1.axhline(y=CONVERGENCE_CRITICAL, color='red', linestyle='--', label=f'Critical ({CONVERGENCE_CRITICAL})')
ax1.axhline(y=0, color='gray', linestyle=':', alpha=0.5)

# Color zones
ax1.axhspan(CONVERGENCE_OPTIMAL, 1.1, alpha=0.1, color='green')
ax1.axhspan(CONVERGENCE_WARNING, CONVERGENCE_OPTIMAL, alpha=0.1, color='yellow')
ax1.axhspan(CONVERGENCE_CRITICAL, CONVERGENCE_WARNING, alpha=0.1, color='orange')
ax1.axhspan(-1.1, CONVERGENCE_CRITICAL, alpha=0.1, color='red')

ax1.set_xlabel('Optimization Step', fontsize=12)
ax1.set_ylabel('Convergence Quality', fontsize=12)
ax1.set_title('MADA Field Direction Correction', fontsize=14, fontweight='bold')
ax1.set_ylim([-1.1, 1.1])
ax1.legend(fontsize=10, loc='lower right')
ax1.grid(True, alpha=0.3)

# Right: Vector field visualization (top view)
# Initial (wrong) arrows
ax2.quiver(mada1_pos[0], mada1_pos[1], 
           B1_dir_history[0][0]*0.2, B1_dir_history[0][1]*0.2,
           color='red', scale=1, width=0.015, label='Initial (WRONG)', alpha=0.7)
ax2.quiver(mada2_pos[0], mada2_pos[1],
           B2_dir_history[0][0]*0.2, B2_dir_history[0][1]*0.2,
           color='red', scale=1, width=0.015, alpha=0.7)

# Final (corrected) arrows
ax2.quiver(mada1_pos[0], mada1_pos[1],
           B1_dir_history[-1][0]*0.2, B1_dir_history[-1][1]*0.2,
           color='green', scale=1, width=0.015, label='Final (CORRECT)', alpha=1.0)
ax2.quiver(mada2_pos[0], mada2_pos[1],
           B2_dir_history[-1][0]*0.2, B2_dir_history[-1][1]*0.2,
           color='green', scale=1, width=0.015, alpha=1.0)

# Mark positions
ax2.scatter(mada1_pos[0], mada1_pos[1], s=200, c='blue', marker='s', label='MADA 1', zorder=5, edgecolors='black')
ax2.scatter(mada2_pos[0], mada2_pos[1], s=200, c='orange', marker='s', label='MADA 2', zorder=5, edgecolors='black')
ax2.scatter(center[0], center[1], s=300, c='gold', marker='*', label='Focal Point', zorder=10, edgecolors='black')

# Draw opposition zone
circle = plt.Circle(center[:2], 0.05, fill=False, color='purple', linestyle='--', linewidth=2, label='High ∇B² Zone')
ax2.add_patch(circle)

ax2.set_xlabel('X Position (m)', fontsize=12)
ax2.set_ylabel('Y Position (m)', fontsize=12)
ax2.set_title('Bushman Opposing Configuration (Top View)', fontsize=14, fontweight='bold')
ax2.set_xlim([-0.5, 0.5])
ax2.set_ylim([-0.3, 0.3])
ax2.legend(fontsize=10, loc='upper left')
ax2.grid(True, alpha=0.3)
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

print("\n📊 Visualization:")
print("   Left: Convergence quality from -1.0 (diverging) to +1.0 (opposing)")
print("   Right: Red arrows = initial wrong directions")
print("          Green arrows = corrected directions toward focal point")
print("          Purple circle = high ∇B² zone where vacuum effects are strongest")

## Part 4: Supra-Saturation Regime Optimization

Find the optimal B_opposing / B_sat ratio for maximum vacuum effect efficiency.

Per the RVG framework:
> The opposing gap field must **substantially exceed material saturation B_s** to achieve macroscopic vacuum effects.

We optimize for different materials: Minnealloy, Hiperco-50, and Iron.

In [ ]:
# =============================================================================
# PART 4: SUPRA-SATURATION REGIME OPTIMIZATION
# =============================================================================

# Material definitions
materials = {
    'Minnealloy': {'B_sat': 2.85, 'color': 'green', 'score': 95},
    'Hiperco-50': {'B_sat': 2.40, 'color': 'blue', 'score': 88},
    'Pure Iron': {'B_sat': 2.10, 'color': 'orange', 'score': 90}
}

# Fixed parameters
GRAD_B2_BASE = 1e10  # Base gradient (T²/m)
VOLUME = 0.1  # m³

# Optimization: Find B_opposing that maximizes thrust efficiency
# Efficiency = Thrust / (Power input ∝ B²)

def calculate_thrust_efficiency(B_opposing, B_sat, theta_base=1e-6, B_crit=20.0):
    """
    Calculate thrust efficiency in supra-saturation regime.
    Efficiency = Thrust / B_opposing² (normalized power)
    """
    # Dilaton enhancement
    theta = theta_base * (1 + (B_opposing / B_crit)**2)
    
    # Supra-saturation effectiveness
    ratio = B_opposing / B_sat
    if ratio > 1.0:
        effectiveness = min(ratio**2 / 25.0, 1.0)
    else:
        effectiveness = ratio / 25.0
    
    # Gradient scales with B_opposing
    grad_B2 = GRAD_B2_BASE * (B_opposing / 50.0)**2
    
    # Thrust
    thrust = (1 / (2 * MU_0)) * theta * effectiveness * grad_B2 * VOLUME * 0.95
    
    # Efficiency (thrust per unit "power" ∝ B²)
    power_proxy = B_opposing**2
    efficiency = thrust / power_proxy
    
    return thrust, efficiency, effectiveness

# Scan B_opposing range
B_range = np.linspace(1, 90, 200)

results = {}
for mat_name, mat_props in materials.items():
    thrusts = []
    efficiencies = []
    effectivenesses = []
    
    for B in B_range:
        T, eff, effectiveness = calculate_thrust_efficiency(B, mat_props['B_sat'])
        thrusts.append(T)
        efficiencies.append(eff)
        effectivenesses.append(effectiveness)
    
    # Find optimal B_opposing
    opt_idx = np.argmax(efficiencies)
    opt_B = B_range[opt_idx]
    opt_ratio = opt_B / mat_props['B_sat']
    
    results[mat_name] = {
        'thrusts': np.array(thrusts),
        'efficiencies': np.array(efficiencies),
        'effectivenesses': np.array(effectivenesses),
        'optimal_B': opt_B,
        'optimal_ratio': opt_ratio,
        'optimal_thrust': thrusts[opt_idx],
        'optimal_efficiency': efficiencies[opt_idx]
    }

print("SUPRA-SATURATION OPTIMIZATION RESULTS")
print("="*70)
print(f"{'Material':<15} {'B_sat (T)':<10} {'Opt B (T)':<12} {'B/B_sat':<10} {'Thrust (N)':<12}")
print("-"*70)
for mat_name, mat_props in materials.items():
    r = results[mat_name]
    print(f"{mat_name:<15} {mat_props['B_sat']:<10.2f} {r['optimal_B']:<12.1f} {r['optimal_ratio']:<10.1f} {r['optimal_thrust']:<12.2e}")

In [ ]:
# Visualization: Supra-saturation analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: Thrust vs B_opposing
for mat_name, mat_props in materials.items():
    r = results[mat_name]
    axes[0].plot(B_range, r['thrusts'], color=mat_props['color'], 
                 linewidth=2, label=f"{mat_name} (B_sat={mat_props['B_sat']}T)")
    # Mark saturation point
    sat_idx = np.argmin(np.abs(B_range - mat_props['B_sat']))
    axes[0].axvline(x=mat_props['B_sat'], color=mat_props['color'], linestyle=':', alpha=0.5)

axes[0].set_xlabel('B_opposing (T)', fontsize=12)
axes[0].set_ylabel('Thrust (N)', fontsize=12)
axes[0].set_title('Thrust vs B-field by Material', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# Middle: Efficiency vs B/B_sat ratio
for mat_name, mat_props in materials.items():
    r = results[mat_name]
    ratio = B_range / mat_props['B_sat']
    axes[1].plot(ratio, r['efficiencies'], color=mat_props['color'],
                 linewidth=2, label=mat_name)
    # Mark optimal point
    axes[1].scatter([r['optimal_ratio']], [r['optimal_efficiency']], 
                   color=mat_props['color'], s=100, zorder=5, edgecolors='black')

axes[1].axvline(x=1.0, color='gray', linestyle='--', label='Saturation (B/B_sat=1)')
axes[1].axvline(x=5.0, color='purple', linestyle='--', alpha=0.5, label='Deep Supra-Sat')
axes[1].set_xlabel('B_opposing / B_sat Ratio', fontsize=12)
axes[1].set_ylabel('Thrust Efficiency (N/T²)', fontsize=12)
axes[1].set_title('Efficiency vs Supra-Saturation Ratio', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Right: Effectiveness curves
for mat_name, mat_props in materials.items():
    r = results[mat_name]
    ratio = B_range / mat_props['B_sat']
    axes[2].plot(ratio, r['effectivenesses'], color=mat_props['color'],
                 linewidth=2, label=mat_name)

axes[2].axvline(x=1.0, color='gray', linestyle='--')
axes[2].axhline(y=0.5, color='orange', linestyle='--', alpha=0.5, label='50% Effectiveness')
axes[2].axhline(y=1.0, color='green', linestyle='--', alpha=0.5, label='Max Effectiveness')
axes[2].set_xlabel('B_opposing / B_sat Ratio', fontsize=12)
axes[2].set_ylabel('Supra-Saturation Effectiveness', fontsize=12)
axes[2].set_title('Vacuum Effect Effectiveness', fontsize=14, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim([0, 30])

plt.tight_layout()
plt.show()

print("\n📊 Key Insights:")
print("   • Thrust increases dramatically in supra-saturation regime (B >> B_sat)")
print("   • Minnealloy allows highest B_opposing before requiring more power")
print("   • Optimal efficiency at B/B_sat ≈ 5-15 depending on material")
print("   • Deep supra-saturation (B/B_sat > 20) maximizes vacuum effects")

## Part 5: Predictive Maintenance with RVG Parameters

Train a neural network for predictive maintenance that includes RVG-specific inputs:

**Inputs (7 features):**
- cycles (operational cycles)
- temp (temperature °C)
- B_field (operating B-field T)
- threat_level (external threat 0-1)
- convergence_quality (MADA alignment 0-1)
- supra_sat_ratio (B/B_sat ratio)
- theta_dilaton (current enhancement factor)

**Outputs:**
- degradation_prob (0-1)
- adapted_frequency (Hz)

In [ ]:
# =============================================================================
# PART 5: PREDICTIVE MAINTENANCE MODEL
# =============================================================================

np.random.seed(42)

# Generate synthetic training data
num_samples = 2000

# Input features
cycles = np.random.randint(1, 10000, num_samples)
temp = np.random.uniform(20, 120, num_samples)
B_field = np.random.uniform(20, 80, num_samples)
threat_level = np.random.uniform(0, 1, num_samples)

# MADA convergence quality (mostly good, some degraded)
convergence_quality = np.random.beta(8, 2, num_samples)
poor_conv_idx = np.random.choice(num_samples, int(0.15 * num_samples), replace=False)
convergence_quality[poor_conv_idx] = np.random.uniform(0.5, 0.8, len(poor_conv_idx))

# Supra-saturation ratio (B/B_sat for Minnealloy)
supra_sat_ratio = B_field / B_SAT_MINNEALLOY

# Theta dilaton (calculated from B_field)
theta_dilaton = DEFAULT_THETA_BASE * (1 + (B_field / DEFAULT_B_CRIT)**2)

# Stack features
X = np.column_stack([
    cycles / 10000,  # Normalize
    temp / 120,
    B_field / 80,
    threat_level,
    convergence_quality,
    supra_sat_ratio / 30,  # Normalize
    theta_dilaton / 1e-4  # Normalize
])

# Generate targets
# Degradation increases with: high cycles, high temp, LOW convergence, HIGH supra-sat stress
base_degradation = 1 / (1 + np.exp(-0.001 * (cycles + 10 * (temp - 50))))
convergence_penalty = (1 - convergence_quality) * 0.4
supra_sat_stress = np.clip((supra_sat_ratio - 10) / 20, 0, 0.3)  # Stress above ratio 10
degradation = np.clip(base_degradation + convergence_penalty + supra_sat_stress, 0, 1)

# Adapted frequency: decreases with degradation and poor convergence
base_freq = 100  # Hz
adapted_freq = base_freq * (1 - 0.5 * degradation) * convergence_quality
adapted_freq += np.random.normal(0, 5, num_samples)  # Noise
adapted_freq = np.clip(adapted_freq, 20, 150)

y = np.column_stack([degradation, adapted_freq / 150])  # Normalize frequency

# Split data
split = int(0.8 * num_samples)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print("PREDICTIVE MAINTENANCE DATASET")
print("="*60)
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"\nInput features (7):")
print("  1. cycles (normalized)")
print("  2. temperature (normalized)")
print("  3. B_field (normalized)")
print("  4. threat_level")
print("  5. convergence_quality")
print("  6. supra_sat_ratio (normalized)")
print("  7. theta_dilaton (normalized)")
print(f"\nOutput targets (2):")
print("  1. degradation_probability")
print("  2. adapted_frequency (normalized)")

In [ ]:
# Build and train model
maintenance_model = Sequential([
    Dense(128, activation='relu', input_shape=(7,)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(2)  # degradation, frequency
])

maintenance_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print("Predictive Maintenance Model:")
maintenance_model.summary()

# Train
print("\nTraining...")
history = maintenance_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.2,
    verbose=0
)

# Evaluate
test_loss, test_mae = maintenance_model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test MAE: {test_mae:.4f}")

In [ ]:
# Visualization: Training history and predictions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: Training curves
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('MSE Loss', fontsize=12)
axes[0].set_title('Training History', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Middle: Predicted vs Actual degradation
y_pred = maintenance_model.predict(X_test, verbose=0)
axes[1].scatter(y_test[:, 0], y_pred[:, 0], alpha=0.3, s=20)
axes[1].plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect Fit')
axes[1].set_xlabel('Actual Degradation', fontsize=12)
axes[1].set_ylabel('Predicted Degradation', fontsize=12)
axes[1].set_title('Degradation Prediction', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Right: Feature importance (via gradient analysis)
feature_names = ['Cycles', 'Temp', 'B_field', 'Threat', 'Convergence', 'Supra-Sat', 'Θ_dilaton']

# Simple gradient-based importance
X_sample = tf.constant(X_test[:100], dtype=tf.float32)
with tf.GradientTape() as tape:
    tape.watch(X_sample)
    pred = maintenance_model(X_sample)
    deg_pred = pred[:, 0]

grads = tape.gradient(deg_pred, X_sample)
importance = np.abs(grads.numpy()).mean(axis=0)
importance = importance / importance.sum()  # Normalize

colors = ['blue' if i != 4 else 'red' for i in range(len(feature_names))]  # Highlight convergence
axes[2].barh(feature_names, importance, color=colors, edgecolor='black')
axes[2].set_xlabel('Relative Importance', fontsize=12)
axes[2].set_title('Feature Importance for Degradation', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n📊 Key Finding: Convergence quality (red bar) is a critical predictor!")
print("   Poor MADA alignment significantly increases degradation risk.")

In [ ]:
# Interactive prediction scenarios
scenarios = [
    {
        'name': '✓ Optimal Operation',
        'input': [5000/10000, 60/120, 50/80, 0.3, 0.98, 17.5/30, 7.25e-6/1e-4],
        'description': 'Normal operation with excellent MADA convergence'
    },
    {
        'name': '⚠️ High Supra-Saturation',
        'input': [5000/10000, 70/120, 70/80, 0.3, 0.95, 24.6/30, 1.3e-5/1e-4],
        'description': 'Deep supra-saturation regime with thermal stress'
    },
    {
        'name': '🔴 Poor Convergence',
        'input': [5000/10000, 60/120, 50/80, 0.3, 0.65, 17.5/30, 7.25e-6/1e-4],
        'description': 'CRITICAL: MADA fields misaligned!'
    },
    {
        'name': '🔴 Multiple Failures',
        'input': [8000/10000, 100/120, 75/80, 0.8, 0.55, 26.3/30, 1.5e-5/1e-4],
        'description': 'High stress + poor convergence + high threat'
    }
]

print("="*70)
print("PREDICTIVE MAINTENANCE SCENARIOS")
print("="*70)

for scenario in scenarios:
    input_data = np.array([scenario['input']])
    prediction = maintenance_model.predict(input_data, verbose=0)
    deg_prob = prediction[0, 0]
    adapt_freq = prediction[0, 1] * 150  # Denormalize
    
    print(f"\n{scenario['name']}")
    print(f"  {scenario['description']}")
    print(f"  Convergence Quality: {scenario['input'][4]:.2f}")
    print(f"  Supra-Sat Ratio: {scenario['input'][5]*30:.1f}")
    print(f"  ────────────────────────────────")
    print(f"  → Degradation Risk: {deg_prob*100:.1f}%")
    print(f"  → Recommended Frequency: {adapt_freq:.1f} Hz")
    
    if deg_prob > 0.7:
        print(f"  → ACTION: Immediate maintenance required!")
    elif deg_prob > 0.4:
        print(f"  → ACTION: Schedule maintenance soon")
    else:
        print(f"  → STATUS: Normal operation")

## Summary: RVG ML Optimization Toolkit

This notebook demonstrated ML applications for the RVG Unified Field framework:

1. **∇B² Gradient Optimization**: Maximize thrust via the Master Equation of Levitation
2. **Θ_dilaton(B) Fitting**: Neural network calibration of the dilaton enhancement function
3. **MADA Convergence**: Automatic detection and correction of field alignment errors
4. **Supra-Saturation Analysis**: Optimal B/B_sat ratios for different materials
5. **Predictive Maintenance**: 7-feature model including RVG-specific parameters

### Key Takeaways:
- **Θ_dilaton(B)** is the critical parameter requiring experimental calibration
- **MADA convergence quality** strongly impacts both thrust and system longevity
- **Supra-saturation** (B >> B_sat) is essential for macroscopic vacuum effects
- **Minnealloy** (B_sat = 2.85 T) enables the highest efficiency operation

### References:
- [RVG Unified Field Theory](https://dx.doi.org/10.2139/ssrn.5381654)
- [U.S. Patent #5,929,732 - MADA](https://patents.google.com/patent/US5929732A/en)

In [ ]:
# Save models for integration with navigation.py and equations.py
print("Saving models...")

try:
    theta_model.save('models/theta_dilaton_model.h5')
    maintenance_model.save('models/maintenance_model.h5')
    print("✓ Models saved to models/ directory")
except Exception as e:
    print(f"Note: Could not save models ({e})")
    print("Models remain in memory for this session.")

print("\n" + "="*70)
print("RVG ML OPTIMIZATION NOTEBOOK COMPLETE")
print("="*70)
print("\nNext steps:")
print("  1. Collect experimental thrust data for Θ_dilaton calibration")
print("  2. Integrate fitted parameters into simulations/equations.py")
print("  3. Update ai/navigation.py with calibrated DEFAULT_THETA_BASE and B_CRIT_EFFECTIVE")
print("  4. Run bench-top experiments per experiments/bench_test_designs.md")